# Auditoría del scraping — Portal Inmobiliario

**Proyecto REAL STATE BIG DATA** · Revisión de archivos **ya descargados**.

**Objetivo:** saber cuántas publicaciones distintas tenemos, comprobar casas/departamentos de la Región Metropolitana y medir el progreso hacia **20.000 IDs únicos**.

**Seguro:** este notebook **no realiza scraping, no se conecta a internet y no modifica los JSON**. Solo los lee.

> Importante: un mismo aviso puede aparecer en varios archivos o en distintas extracciones. Lo contamos una sola vez por `id` y mantenemos su observación más reciente para revisar calidad. No sumamos los archivos como si todas fueran publicaciones nuevas.

## 1. Configuración y ubicación de los JSON

Abre este notebook desde la carpeta `notebooks/` del proyecto. Detecta automáticamente la raíz del repositorio. Si lo colocas en otra carpeta, modifica `RUTA_DATOS` manualmente.

In [15]:
from pathlib import Path
import json
import re
import pandas as pd

META = 20_000
CATEGORIAS_OBJETIVO = {"casa", "departamento"}
REGION_OBJETIVO = "metropolitana"  # Cambiar a None si después ampliamos a todo Chile

ubicacion_actual = Path.cwd().resolve()
raices = [ubicacion_actual, *ubicacion_actual.parents]
carpeta_encontrada = next(
    (r / "datalake_crudo" / "portal_inmobiliario"
     for r in raices
     if (r / "datalake_crudo" / "portal_inmobiliario").is_dir()),
    None,
)
RUTA_DATOS = carpeta_encontrada
if RUTA_DATOS is None:
    raise FileNotFoundError(
        "No encontré datalake_crudo/portal_inmobiliario desde la carpeta actual. "
        "Abre el notebook dentro del repositorio o fija RUTA_DATOS manualmente."
    )

print("Carpeta analizada:", RUTA_DATOS)
print("Meta de publicaciones únicas:", f"{META:,}".replace(",", "."))

Carpeta analizada: C:\Users\felip\Documents\GitHub\Proyecto_inmobiliario\datalake_crudo\portal_inmobiliario
Meta de publicaciones únicas: 20.000


## 2. Inventario y lectura de todos los JSON

Leemos todos los archivos `portal*.json`, incluidos los antiguos y los del scraper ampliado. Si un archivo está corrupto o no contiene una lista de publicaciones, lo señalamos sin detener la auditoría.

In [16]:
def registros_desde_json(objeto):
    if isinstance(objeto, list):
        return objeto
    if isinstance(objeto, dict):
        for clave in ("publicaciones", "registros", "datos", "items", "results"):
            if isinstance(objeto.get(clave), list):
                return objeto[clave]
    return None

archivos = sorted(RUTA_DATOS.rglob("portal*.json"))
filas = []
inventario = []
advertencias = []

for archivo in archivos:
    try:
        with archivo.open(encoding="utf-8") as f:
            contenido = json.load(f)
        registros = registros_desde_json(contenido)
        if registros is None:
            advertencias.append(f"Formato no reconocido: {archivo.name}")
            continue
        n_validos = 0
        for registro in registros:
            if not isinstance(registro, dict):
                continue
            nuevo = dict(registro)
            nuevo["_archivo"] = archivo.name
            nuevo["_archivo_modificado"] = archivo.stat().st_mtime
            filas.append(nuevo)
            n_validos += 1
        inventario.append({"archivo": archivo.name, "registros": n_validos,
                           "tamano_kb": round(archivo.stat().st_size / 1024, 1)})
    except (OSError, UnicodeError, json.JSONDecodeError) as exc:
        advertencias.append(f"No se pudo leer {archivo.name}: {exc}")

df_archivos = pd.DataFrame(inventario)
df = pd.DataFrame(filas)
print("Archivos JSON encontrados:", len(archivos))
print("Archivos con registros legibles:", len(inventario))
print("Registros leídos (incluyen repetidos):", len(df))
if advertencias:
    print("\nADVERTENCIAS (primeras 10):")
    for msg in advertencias[:10]:
        print("-", msg)
if df.empty:
    raise ValueError("No hay registros para analizar. Revisa la carpeta y los nombres de los JSON.")
display(df_archivos.sort_values("archivo").tail(10))

Archivos JSON encontrados: 34
Archivos con registros legibles: 34
Registros leídos (incluyen repetidos): 8695


,archivo,registros,tamano_kb
24,portal_venta_departamento_santiago-metropolita...,192,126.5
25,portal_venta_departamento_santiago-metropolita...,192,155.4
26,portal_venta_departamentos_santiago_20261022_0...,192,126.5
27,portal_venta_departamentos_santiago_20261027_2...,192,125.6
28,portal_venta_industrial_metropolitana_sin_moda...,192,144.9
29,portal_venta_oficina_metropolitana_sin_modalid...,192,147.0
30,portal_venta_oficina_metropolitana_sin_modalid...,192,147.0
31,portal_venta_oficina_valparaiso_sin_modalidad_...,192,144.5
32,portal_venta_sitio_metropolitana_sin_modalidad...,192,137.1
33,portal_venta_sitio_valparaiso_sin_modalidad_20...,192,136.6


## 3. Normalizar identificadores, categoría y región

Los scrapers anteriores pueden guardar variantes en los nombres. Tomamos primero los campos del registro y, si faltan, intentamos inferir tipo y región desde el **nombre del archivo**. Lo que no pueda identificarse queda como `sin_informacion`, en lugar de inventarlo.

In [17]:
def texto(valor):
    return "" if valor is None else str(valor).strip()

def id_publicacion(registro):
    candidato = texto(registro.get("id") or registro.get("id_publicacion"))
    if candidato and candidato.lower() not in {"nan", "none", "null"}:
        coincidencia = re.search(r"MLC[-_ ]?(\d+)", candidato, re.IGNORECASE)
        return coincidencia.group(1) if coincidencia else candidato
    url = texto(registro.get("url"))
    coincidencia = re.search(r"MLC[-_ ]?(\d+)", url, re.IGNORECASE)
    return coincidencia.group(1) if coincidencia else None

def tipo_normalizado(registro):
    valor = texto(registro.get("tipo_propiedad_busqueda") or registro.get("tipo_propiedad"))
    nombre = texto(registro.get("_archivo")).casefold()
    if not valor:
        patron = re.search(r"(?:venta|arriendo)_(casa|departamento|oficina|industrial|comercial|sitio)_", nombre)
        valor = patron.group(1) if patron else ""
    valor = valor.casefold()
    if "departamento" in valor or valor == "depto":
        return "departamento"
    if "casa" in valor:
        return "casa"
    return valor or "sin_informacion"

def region_normalizada(registro):
    valor = texto(registro.get("ubicacion_busqueda") or registro.get("region_busqueda") or registro.get("region"))
    nombre = texto(registro.get("_archivo"))
    conjunto = (valor + " " + nombre).casefold()
    if "metropolitana" in conjunto or "santiago-metropolitana" in conjunto:
        return "metropolitana"
    if not valor:
        return "sin_informacion"
    return valor.casefold().replace("_", "-")

filas_normalizadas = df.to_dict(orient="records")
df["_id"] = [id_publicacion(r) for r in filas_normalizadas]
df["_tipo"] = [tipo_normalizado(r) for r in filas_normalizadas]
df["_region"] = [region_normalizada(r) for r in filas_normalizadas]

# Considerar la fecha de la publicación, usando la fecha del archivo como respaldo.
fechas_registros = pd.to_datetime(df.get("fecha_extraccion", pd.Series(index=df.index, dtype="object")),
                                  errors="coerce", utc=True, format="mixed")
fechas_archivos = pd.to_datetime(df["_archivo_modificado"], unit="s", utc=True)
df["_momento"] = fechas_registros.fillna(fechas_archivos)

sin_id = int(df["_id"].isna().sum())
con_id = df.loc[df["_id"].notna()].copy()
unicos = (con_id.sort_values(["_momento", "_archivo"])
          .drop_duplicates(subset=["_id"], keep="last").copy())

print("Registros sin identificador:", sin_id)
print("Observaciones con ID:", len(con_id))
print("Publicaciones únicas globales:", len(unicos))
print("Apariciones repetidas de IDs entre archivos:", len(con_id) - len(unicos))
print("Nota: las apariciones repetidas pueden ser capturas históricas del mismo aviso.")

Registros sin identificador: 0
Observaciones con ID: 8695
Publicaciones únicas globales: 7609
Apariciones repetidas de IDs entre archivos: 1086
Nota: las apariciones repetidas pueden ser capturas históricas del mismo aviso.


## 4. Progreso de la meta: casas y departamentos en la RM

Esta es la cifra que usaremos para evaluar si nos acercamos a **20.000 publicaciones distintas**, con la cobertura elegida. Cambiar de región o tipo cambia el denominador.

In [18]:
objetivo = unicos[unicos["_tipo"].isin(CATEGORIAS_OBJETIVO)].copy()
if REGION_OBJETIVO is not None:
    objetivo = objetivo[objetivo["_region"] == REGION_OBJETIVO].copy()

cuenta = len(objetivo)
porcentaje = 100 * cuenta / META if META else 0
faltan = max(META - cuenta, 0)
print("=== OBJETIVO DE EXTRACCIÓN ===")
print("Tipos:", ", ".join(sorted(CATEGORIAS_OBJETIVO)))
print("Región:", REGION_OBJETIVO or "todas")
print("Publicaciones únicas válidas:", f"{cuenta:,}".replace(",", "."))
print("Meta:", f"{META:,}".replace(",", "."))
print(f"Progreso: {porcentaje:.1f}%")
print("Faltan:", f"{faltan:,}".replace(",", "."))
print("\nPor tipo de propiedad:")
display(objetivo["_tipo"].value_counts().rename_axis("tipo").reset_index(name="publicaciones_unicas"))

=== OBJETIVO DE EXTRACCIÓN ===
Tipos: casa, departamento
Región: metropolitana
Publicaciones únicas válidas: 4.240
Meta: 20.000
Progreso: 21.2%
Faltan: 15.760

Por tipo de propiedad:


,tipo,publicaciones_unicas
0,casa,2205
1,departamento,2035


## 5. Cobertura del conjunto completo

Este resumen permite identificar si estamos extrayendo muchos anuncios de una sola categoría o de una sola región.

In [19]:
print("Publicaciones únicas por región y tipo (primeros 30 grupos):")
cobertura = (unicos.groupby(["_region", "_tipo"]).size()
             .reset_index(name="publicaciones_unicas")
             .sort_values("publicaciones_unicas", ascending=False))
display(cobertura.head(30))
print("Publicaciones sin categoría identificable:", int((unicos["_tipo"] == "sin_informacion").sum()))
print("Publicaciones sin región identificable:", int((unicos["_region"] == "sin_informacion").sum()))

Publicaciones únicas por región y tipo (primeros 30 grupos):


,_region,_tipo,publicaciones_unicas
1,metropolitana,casa,2205
3,metropolitana,departamento,2035
5,metropolitana,oficina,627
2,metropolitana,comercial,599
6,metropolitana,sitio,598
4,metropolitana,industrial,582
7,nan,departamento,195
0,biobio,casa,192
8,valparaiso,comercial,192
9,valparaiso,oficina,192


Publicaciones sin categoría identificable: 0
Publicaciones sin región identificable: 0


## 6. Calidad de los registros únicos

**Valores faltantes:** una publicación puede tener ID válido, pero carecer de título, precio o ubicación. Estos porcentajes se calculan sobre el subconjunto objetivo, sin alterar ningún archivo.

In [20]:
def vacio(valor):
    if valor is None:
        return True
    if isinstance(valor, (list, dict, tuple, set)):
        return len(valor) == 0
    try:
        if bool(pd.isna(valor)):
            return True
    except (ValueError, TypeError):
        pass
    return isinstance(valor, str) and valor.strip() == ""

if objetivo.empty:
    print("Todavía no hay registros únicos en el subconjunto objetivo.")
else:
    campos = ["titulo", "precio_actual", "ubicacion", "url", "atributos"]
    faltantes = []
    for campo in campos:
        cantidad = (objetivo[campo].map(vacio).sum() if campo in objetivo.columns else len(objetivo))
        faltantes.append({"campo": campo, "faltantes": int(cantidad),
                          "porcentaje": round(100 * cantidad / len(objetivo), 2)})
    display(pd.DataFrame(faltantes))
    muestra = [c for c in ("_id", "_tipo", "ubicacion", "precio_actual", "titulo", "_archivo") if c in objetivo.columns]
    print("Muestra de registros únicos (observación más reciente):")
    display(objetivo[muestra].head(5))

,campo,faltantes,porcentaje
0,titulo,0,0.0
1,precio_actual,0,0.0
2,ubicacion,0,0.0
3,url,0,0.0
4,atributos,0,0.0


Muestra de registros únicos (observación más reciente):


,_id,_tipo,ubicacion,precio_actual,titulo,_archivo
6776,4530701812,departamento,"Av. Vicuña Mackenna 1281, Ñuñoa, Santiago, Bog...",UF 1.877,Departamento En Venta En Vicuña Mackenna Comun...,portal_venta_departamento_santiago-metropolita...
6777,4509236084,departamento,"Av. Pdte. Balmaceda 2610, Santiago, Parque Los...",UF 2.991,Venta Departamento Estilo Mariposa Parque Los ...,portal_venta_departamento_santiago-metropolita...
6785,2340719939,departamento,"OPORTUNIDAD EN ÑUÑOA / 2D / A PASOS DEL METRO,...",UF 3.100,Oportunidad En Ñuñoa / 2d / A Pasos Del Metro,portal_venta_departamento_santiago-metropolita...
6788,4373692394,departamento,"Amunátegui, Catedral... Metro Santa Ana, Centr...",UF 1.690,"Venta Amplio Depto. Studio, Metros Santa Ana Y...",portal_venta_departamento_santiago-metropolita...
6790,4505447790,departamento,"San Ignacio De Loyola 999, Santiago, Parque Al...",$ 120.000.000,Departamento 1er. Piso Con Amplio Patio Privad...,portal_venta_departamento_santiago-metropolita...


## 7. Revisar los archivos recientes

Sirve para observar si nuevas extracciones están aumentando el total. La **suma de los registros por archivo no representa el número de anuncios únicos**.

In [21]:
if not df_archivos.empty:
    print("Últimos archivos según su fecha de modificación:")
    archivos_con_fecha = df_archivos.copy()
    archivos_con_fecha["fecha_modificacion"] = archivos_con_fecha["archivo"].map(
        {p.name: pd.Timestamp(p.stat().st_mtime, unit="s", tz="UTC") for p in archivos}
    )
    display(archivos_con_fecha.sort_values("fecha_modificacion", ascending=False).head(15))

print("\nRESUMEN FINAL")
print(f"{len(archivos)} archivos; {len(df)} registros leídos; {len(unicos)} IDs únicos globales.")
print(f"Casas + departamentos ({REGION_OBJETIVO or 'todas las regiones'}): {len(objetivo)} IDs únicos.")
print("\nPara seguir creciendo, añade nuevas búsquedas permitidas, no simplemente repitas la misma paginación.")

Últimos archivos según su fecha de modificación:


,archivo,registros,tamano_kb,fecha_modificacion
14,portal_regional_venta_departamento_metropolita...,318,263.5,2026-11-02 01:09:07.860641718+00:00
10,portal_regional_venta_casa_metropolitana_20261...,332,273.8,2026-11-02 01:07:59.250503301+00:00
13,portal_regional_venta_departamento_metropolita...,231,192.2,2026-11-02 01:06:38.631583214+00:00
9,portal_regional_venta_casa_metropolitana_20261...,238,195.0,2026-11-02 01:05:55.678635597+00:00
12,portal_regional_venta_departamento_metropolita...,765,605.8,2026-11-02 00:54:30.485040188+00:00
8,portal_regional_venta_casa_metropolitana_20261...,861,698.2,2026-11-02 00:50:10.177886963+00:00
11,portal_regional_venta_departamento_metropolita...,96,75.6,2026-11-02 00:44:37.110334396+00:00
7,portal_ampliado_venta_sitio_metropolitana_sin_...,406,292.5,2026-11-01 02:01:55.004877806+00:00
2,portal_ampliado_venta_comercial_metropolitana_...,407,305.3,2026-11-01 02:00:11.350276709+00:00
5,portal_ampliado_venta_industrial_metropolitana...,390,289.4,2026-11-01 01:58:28.383328676+00:00



RESUMEN FINAL
34 archivos; 8695 registros leídos; 7609 IDs únicos globales.
Casas + departamentos (metropolitana): 4240 IDs únicos.

Para seguir creciendo, añade nuevas búsquedas permitidas, no simplemente repitas la misma paginación.


## 8. Interpretación y decisiones

- Si los **ID únicos** aumentan, la extracción está ampliando la cobertura.
- Si aumentan los **registros totales** pero no los ID únicos, probablemente estamos volviendo a encontrar anuncios existentes.
- Si falta mucha información de precio o ubicación, debemos revisar el scraper antes de cargar datos al Warehouse.
- **No inferir ventas:** las publicaciones corresponden a **precios de oferta**, no transacciones.
- **La meta de 20.000 no está garantizada**: depende de anuncios accesibles, filtros, paginación y restricciones del portal.

**Próximo paso:** con los resultados de este notebook, decidir cómo ampliar los filtros de casas y departamentos sin generar duplicaciones ni solicitudes innecesarias.